In [1]:
from pyflink.table import EnvironmentSettings, TableEnvironment
from pyflink.table.expressions import col

# Configure Flink environment
env_settings = EnvironmentSettings.in_streaming_mode()
table_env = TableEnvironment.create(env_settings)

ModuleNotFoundError: No module named 'pyflink'

In [ ]:
from pyflink.table import EnvironmentSettings, TableEnvironment
from pyflink.table.expressions import col

# Configure Flink environment
env_settings = EnvironmentSettings.in_streaming_mode()
table_env = TableEnvironment.create(env_settings)

# Define input table (test-topic)
table_env.execute_sql("""
    CREATE TABLE transactions (
        name STRING,
        email STRING,
        address STRING,
        `timestamp` STRING,
        country STRING,
        currency STRING,
        value DOUBLE,
        transaction_date STRING,
        item_description STRING,
        transaction_id STRING
    ) WITH (
        'connector' = 'kafka',
        'topic' = 'test-topic',
        'properties.bootstrap.servers' = 'kafka:9092',
        'properties.group.id' = 'flink-aggregation-group',
        'scan.startup.mode' = 'earliest-offset',
        'format' = 'json'
    )
""")

# Define output table (aggregated-topic)
table_env.execute_sql("""
    CREATE TABLE aggregates (
        data_apenas STRING,
        moeda STRING,
        pais STRING,
        quantidade_registros BIGINT,
        soma_value DOUBLE
    ) WITH (
        'connector' = 'kafka',
        'topic' = 'aggregated-topic',
        'properties.bootstrap.servers' = 'kafka:9092',
        'format' = 'json'
    )
""")

# Load input table
table = table_env.from_path("transactions")

# Perform aggregation
result = table.group_by(
    col("transaction_date").alias("data_apenas"),
    col("currency").alias("moeda"),
    col("country").alias("pais")
).select(
    col("data_apenas"),
    col("moeda"),
    col("pais"),
    table.count().alias("quantidade_registros"),
    col("value").sum().alias("soma_value")
)

# Write to aggregated-topic
result.execute_insert("aggregates").wait()

# Execute job
table_env.execute("Transaction Aggregation")